# MINOS 2x2 Interactive 3D Event Display (Matplotlib)

This notebook renders a **$2 \times 2$ grid of 3D hit displays** for a MINOS event using Matplotlib (`mpl_toolkits.mplot3d`).

### 2x2 Subplot Layout:
- **Top-Left**: Planeview 2 (U View) — East Readout (`ph0.pe` & `time0`)
- **Top-Right**: Planeview 2 (U View) — West Readout (`ph1.pe` & `time1`)
- **Bottom-Left**: Planeview 3 (V View) — East Readout (`ph0.pe` & `time0`)
- **Bottom-Right**: Planeview 3 (V View) — West Readout (`ph1.pe` & `time1`)

- **Synchronized 3D Controls**: The **Elevation (`elev`)** and **Azimuth (`azim`)** sliders rotate all four 3D subplots simultaneously.


In [1]:
# Print all available branches in the ROOT tree
from pathlib import Path
import uproot

rf_path = Path("f21048000_0000_L010185N_D07_r3.sntp.dogwood5.0.root")
if not rf_path.exists():
    rf_path = Path("minos_sparse/f21048000_0000_L010185N_D07_r3.sntp.dogwood5.0.root")

rf = uproot.open(rf_path)
tree_key = "NtpSt;1" if "NtpSt;1" in rf else ("NtpSt" if "NtpSt" in rf else list(rf.keys())[0])
tree = rf[tree_key]

print(f"=== Tree '{tree_key}' contains {len(tree.keys())} branches ===")
print()
for branch_name in sorted(tree.keys()):
    print(branch_name)


=== Tree 'NtpSt;1' contains 942 branches ===

NtpStRecord
NtpStRecord/RecRecordImp<RecCandHeader>
NtpStRecord/RecRecordImp<RecCandHeader>/RecRecord
NtpStRecord/RecRecordImp<RecCandHeader>/RecRecord/TNamed
NtpStRecord/RecRecordImp<RecCandHeader>/RecRecord/TNamed/TObject
NtpStRecord/RecRecordImp<RecCandHeader>/RecRecord/TNamed/TObject/fBits
NtpStRecord/RecRecordImp<RecCandHeader>/RecRecord/TNamed/TObject/fUniqueID
NtpStRecord/RecRecordImp<RecCandHeader>/RecRecord/TNamed/fName
NtpStRecord/RecRecordImp<RecCandHeader>/RecRecord/TNamed/fTitle
NtpStRecord/RecRecordImp<RecCandHeader>/fHeader.RecPhysicsHeader
NtpStRecord/RecRecordImp<RecCandHeader>/fHeader.RecPhysicsHeader/fHeader.RecDataHeader
NtpStRecord/RecRecordImp<RecCandHeader>/fHeader.RecPhysicsHeader/fHeader.RecDataHeader/fHeader.RecHeader
NtpStRecord/RecRecordImp<RecCandHeader>/fHeader.RecPhysicsHeader/fHeader.RecDataHeader/fHeader.RecHeader/fHeader.TObject
NtpStRecord/RecRecordImp<RecCandHeader>/fHeader.RecPhysicsHeader/fHeader.RecDat

In [7]:
%pip install -q uproot awkward numpy matplotlib ipywidgets ipympl

from pathlib import Path
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, IntSlider
import uproot

root_file = Path('f21048000_0000_L010185N_D07_r3.sntp.dogwood5.0.root')
if not root_file.exists():
    root_file = Path('minos_sparse/f21048000_0000_L010185N_D07_r3.sntp.dogwood5.0.root')
assert root_file.exists(), f'Missing ROOT file: {root_file}'
print(f'Using ROOT file: {root_file}')


Note: you may need to restart the kernel to use updated packages.
Using ROOT file: f21048000_0000_L010185N_D07_r3.sntp.dogwood5.0.root


In [8]:
EVENT_INDEX = 0

root_handle = uproot.open(root_file)
tree = root_handle['NtpSt;1']
print(f'Loaded tree NtpSt;1 with {tree.num_entries} total events.')


Loaded tree NtpSt;1 with 119205 total events.


In [11]:
NEUTRINO_FLAVORS = {14: "nu_mu", -14: "nu_bar_mu", 12: "nu_e", -12: "nu_bar_e", 16: "nu_tau", -16: "nu_bar_tau"}
INTERACTION_CURRENTS = {1: "CC", 0: "NC"}
INTERACTION_CHANNELS = {1001: "QE", 1002: "RES", 1003: "DIS", 1004: "COH"}

req_branches = [
    "NtpStRecord/stp/stp.planeview",
    "NtpStRecord/stp/stp.strip",
    "NtpStRecord/stp/stp.plane",
    "NtpStRecord/stp/stp.ph0.pe",
    "NtpStRecord/stp/stp.ph1.pe",
    "NtpStRecord/stp/stp.time0",
    "NtpStRecord/stp/stp.time1",
    "NtpStRecord/mc/mc.inu",
    "NtpStRecord/mc/mc.iaction",
    "NtpStRecord/mc/mc.iresonance",
    "NtpStRecord/mc/mc.p4neu[4]",
]

# Extract event arrays
event = tree.arrays(req_branches, entry_start=EVENT_INDEX, entry_stop=EVENT_INDEX+1, library="ak")[0]

planeview = ak.to_list(event["NtpStRecord/stp/stp.planeview"])
strip = ak.to_list(event["NtpStRecord/stp/stp.strip"])
plane = ak.to_list(event["NtpStRecord/stp/stp.plane"])
ph0_pe = ak.to_list(event["NtpStRecord/stp/stp.ph0.pe"])
ph1_pe = ak.to_list(event["NtpStRecord/stp/stp.ph1.pe"])
time0 = ak.to_list(event["NtpStRecord/stp/stp.time0"])
time1 = ak.to_list(event["NtpStRecord/stp/stp.time1"])

# Print MC Truth info
try:
    inu_val = ak.to_list(event["NtpStRecord/mc/mc.inu"])[0]
    iact_val = ak.to_list(event["NtpStRecord/mc/mc.iaction"])[0]
    ires_val = ak.to_list(event["NtpStRecord/mc/mc.iresonance"])[0]
    p4neu_val = ak.to_list(event["NtpStRecord/mc/mc.p4neu[4]"])[0]
    nu_str = NEUTRINO_FLAVORS.get(inu_val, f"PDG {inu_val}")
    curr_str = INTERACTION_CURRENTS.get(iact_val, f"Current {iact_val}")
    chan_str = INTERACTION_CHANNELS.get(ires_val, f"Code {ires_val}")
    e_nu_val = p4neu_val[3]
    print(f"Event {EVENT_INDEX} MC Truth: {nu_str} {curr_str} {chan_str} | True E_nu = {e_nu_val:.2f} GeV")
except Exception as e:
    print(f"Event {EVENT_INDEX} MC Truth unavailable.")

def render_2x2_3d_event(elevation=25, azimuth=-60):
    fig = plt.figure(figsize=(16, 12), facecolor="#111111")
    fig.suptitle(f"MINOS 3D Event Display (Event {EVENT_INDEX})", color="white", fontsize=16, y=0.98)

    configs = [
        (1, 2, "Planeview 2 (U View) - East Readout (ph0.pe & time0)", time0, ph0_pe),
        (2, 2, "Planeview 2 (U View) - West Readout (ph1.pe & time1)", time1, ph1_pe),
        (3, 3, "Planeview 3 (V View) - East Readout (ph0.pe & time0)", time0, ph0_pe),
        (4, 3, "Planeview 3 (V View) - West Readout (ph1.pe & time1)", time1, ph1_pe),
    ]

    for idx, target_pv, title, t_arr, ph_arr in configs:
        ax = fig.add_subplot(2, 2, idx, projection="3d", facecolor="#111111")
        ax.view_init(elev=elevation, azim=azimuth)

        mask = [(pv == target_pv and t > -900000) for pv, t in zip(planeview, t_arr)]
        sub_plane = np.array([p for p, keep in zip(plane, mask) if keep])
        sub_strip = np.array([s for s, keep in zip(strip, mask) if keep])
        sub_time_us = np.array([t * 1e6 for t, keep in zip(t_arr, mask) if keep])
        sub_ph = np.array([ph for ph, keep in zip(ph_arr, mask) if keep])

        if len(sub_plane) > 0:
            sc = ax.scatter(
                sub_plane, sub_strip, sub_time_us,
                c=sub_ph, cmap="turbo", s=35, edgecolors="none", alpha=0.9
            )
            cbar = fig.colorbar(sc, ax=ax, pad=0.08, shrink=0.6)
            cbar.set_label("Pulse Height [PE]", color="white")
            cbar.ax.yaxis.set_tick_params(color="white")
            plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

        ax.set_xlabel("stp.plane", color="white", labelpad=8)
        ax.set_ylabel("stp.strip", color="white", labelpad=8)
        ax.set_zlabel("time [us]", color="white", labelpad=8)
        ax.tick_params(colors="white")
        ax.xaxis.pane.fill = False
        ax.yaxis.pane.fill = False
        ax.zaxis.pane.fill = False
        ax.xaxis.pane.set_edgecolor("#444444")
        ax.yaxis.pane.set_edgecolor("#444444")
        ax.zaxis.pane.set_edgecolor("#444444")
        ax.set_title(title, color="white", fontsize=12)

    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()

# Create interactive sliders controlling all 4 subplots simultaneously
interact(
    render_2x2_3d_event,
    elevation=IntSlider(min=-90, max=90, step=5, value=25, description='Elevation°'),
    azimuth=IntSlider(min=-180, max=180, step=5, value=-60, description='Azimuth°')
);


Event 0 MC Truth: nu_mu CC RES | True E_nu = 2.76 GeV


interactive(children=(IntSlider(value=25, description='Elevation°', max=90, min=-90, step=5), IntSlider(value=…

In [10]:
# import awkward as ak
# import matplotlib.pyplot as plt

# # Optional interactive widget import:
# # from google.colab import output; output.enable_custom_widget_manager() # If using Google Colab
# from ipywidgets import IntSlider, interact
# import numpy as np

# NEUTRINO_FLAVORS = {
#     14: "nu_mu",
#     -14: "nu_bar_mu",
#     12: "nu_e",
#     -12: "nu_bar_e",
#     16: "nu_tau",
#     -16: "nu_bar_tau",
# }
# INTERACTION_CURRENTS = {1: "CC", 0: "NC"}
# INTERACTION_CHANNELS = {1001: "QE", 1002: "RES", 1003: "DIS", 1004: "COH"}

# # Target raw digihit branches
# req_branches = [
#     "NtpStRecord/digihit/digihit.planeview",
#     "NtpStRecord/digihit/digihit.strip",
#     "NtpStRecord/digihit/digihit.plane",
#     "NtpStRecord/digihit/digihit.pE",
#     "NtpStRecord/digihit/digihit.t0",
#     "NtpStRecord/digihit/digihit.t1",
#     "NtpStRecord/mc/mc.inu",
#     "NtpStRecord/mc/mc.iaction",
#     "NtpStRecord/mc/mc.iresonance",
#     "NtpStRecord/mc/mc.p4neu[4]",
# ]

# # Extract event arrays from tree
# event = tree.arrays(
#     req_branches,
#     entry_start=EVENT_INDEX,
#     entry_stop=EVENT_INDEX + 1,
#     library="ak",
# )[0]

# planeview = ak.to_list(event["NtpStRecord/digihit/digihit.planeview"])
# strip = ak.to_list(event["NtpStRecord/digihit/digihit.strip"])
# plane = ak.to_list(event["NtpStRecord/digihit/digihit.plane"])
# pE = ak.to_list(event["NtpStRecord/digihit/digihit.pE"])
# t0 = ak.to_list(event["NtpStRecord/digihit/digihit.t0"])
# t1 = ak.to_list(event["NtpStRecord/digihit/digihit.t1"])

# # Print MC Truth info
# try:
#     inu_val = ak.to_list(event["NtpStRecord/mc/mc.inu"])[0]
#     iact_val = ak.to_list(event["NtpStRecord/mc/mc.iaction"])[0]
#     ires_val = ak.to_list(event["NtpStRecord/mc/mc.iresonance"])[0]
#     p4neu_val = ak.to_list(event["NtpStRecord/mc/mc.p4neu[4]"])[0]
#     nu_str = NEUTRINO_FLAVORS.get(inu_val, f"PDG {inu_val}")
#     curr_str = INTERACTION_CURRENTS.get(iact_val, f"Current {iact_val}")
#     chan_str = INTERACTION_CHANNELS.get(ires_val, f"Code {ires_val}")
#     e_nu_val = p4neu_val[3]
#     print(
#         f"Event {EVENT_INDEX} MC Truth: {nu_str} {curr_str} {chan_str} | True E_nu = {e_nu_val:.2f} GeV"
#     )
# except Exception as e:
#     print(f"Event {EVENT_INDEX} MC Truth unavailable.")


# def render_2x2_3d_digihit_event(elevation=25, azimuth=-60):
#     fig = plt.figure(figsize=(16, 12), facecolor="#111111")
#     fig.suptitle(
#         f"MINOS Raw Digihit 3D Event Display (Event {EVENT_INDEX})",
#         color="white",
#         fontsize=16,
#         y=0.98,
#     )

#     # 4 Subplot panel configurations matching U/V views and t0/t1 readout ends
#     configs = [
#         (
#             1,
#             2,
#             "Planeview 2 (U View) - End 0 Readout (pE & t0)",
#             t0,
#             pE,
#         ),
#         (
#             2,
#             2,
#             "Planeview 2 (U View) - End 1 Readout (pE & t1)",
#             t1,
#             pE,
#         ),
#         (
#             3,
#             3,
#             "Planeview 3 (V View) - End 0 Readout (pE & t0)",
#             t0,
#             pE,
#         ),
#         (
#             4,
#             3,
#             "Planeview 3 (V View) - End 1 Readout (pE & t1)",
#             t1,
#             pE,
#         ),
#     ]

#     for idx, target_pv, title, t_arr, pe_arr in configs:
#         ax = fig.add_subplot(2, 2, idx, projection="3d", facecolor="#111111")
#         ax.view_init(elev=elevation, azim=azimuth)

#         # Filter unphysical times (default unset hit entries)
#         mask = [
#             (pv == target_pv and t > -900000)
#             for pv, t in zip(planeview, t_arr)
#         ]
#         sub_plane = np.array([p for p, keep in zip(plane, mask) if keep])
#         sub_strip = np.array([s for s, keep in zip(strip, mask) if keep])
#         sub_time_us = np.array(
#             [t * 1e6 for t, keep in zip(t_arr, mask) if keep]
#         )
#         sub_pe = np.array([pe for pe, keep in zip(pe_arr, mask) if keep])

#         if len(sub_plane) > 0:
#             sc = ax.scatter(
#                 sub_plane,
#                 sub_strip,
#                 sub_time_us,
#                 c=sub_pe,
#                 cmap="turbo",
#                 s=35,
#                 edgecolors="none",
#                 alpha=0.9,
#             )
#             cbar = fig.colorbar(sc, ax=ax, pad=0.08, shrink=0.6)
#             cbar.set_label("Raw Pulse Height [pE]", color="white")
#             cbar.ax.yaxis.set_tick_params(color="white")
#             plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")

#         ax.set_xlabel("digihit.plane", color="white", labelpad=8)
#         ax.set_ylabel("digihit.strip", color="white", labelpad=8)
#         ax.set_zlabel("time [us]", color="white", labelpad=8)
#         ax.tick_params(colors="white")
#         ax.xaxis.pane.fill = False
#         ax.yaxis.pane.fill = False
#         ax.zaxis.pane.fill = False
#         ax.xaxis.pane.set_edgecolor("#444444")
#         ax.yaxis.pane.set_edgecolor("#444444")
#         ax.zaxis.pane.set_edgecolor("#444444")
#         ax.set_title(title, color="white", fontsize=12)

#     plt.tight_layout()
#     plt.subplots_adjust(top=0.93)
#     plt.show()


# # Create interactive sliders controlling all 4 subplots simultaneously
# interact(
#     render_2x2_3d_digihit_event,
#     elevation=IntSlider(
#         min=-90, max=90, step=5, value=25, description="Elevation°"
#     ),
#     azimuth=IntSlider(
#         min=-180, max=180, step=5, value=-60, description="Azimuth°"
#     ),
# );

Event 0 MC Truth: nu_mu CC RES | True E_nu = 2.76 GeV


interactive(children=(IntSlider(value=25, description='Elevation°', max=90, min=-90, step=5), IntSlider(value=…